In [0]:
from pyspark.sql.functions import (
    count,
    countDistinct,
    avg,
    min,
    max,
    round,
    current_timestamp,
    desc
)

In [0]:
# Parámetros
catalog_name = "catalog"

silver_schema = "silver"
gold_schema = "gold"

silver_table = (
    f"{catalog_name}.{silver_schema}.automobile_enriched"
)

gold_kpis_table = (
    f"{catalog_name}.{gold_schema}.automobile_kpis"
)

gold_by_make_table = (
    f"{catalog_name}.{gold_schema}.automobile_by_make"
)

gold_by_fuel_table = (
    f"{catalog_name}.{gold_schema}.automobile_by_fuel_type"
)

gold_top_expensive_table = (
    f"{catalog_name}.{gold_schema}.top_expensive_cars"
)

In [0]:
# Leer tablas Silver
df_silver = spark.table(
    silver_table
)

assert df_silver.count() > 0

In [0]:
df_gold_kpis = (
    df_silver
    .agg(

        count("*").alias(
            "total_vehicles"
        ),

        countDistinct(
            "manufacturer_id"
        ).alias(
            "total_manufacturers"
        ),

        round(
            avg("price"), 2
        ).alias(
            "average_price"
        ),

        min("price").alias(
            "minimum_price"
        ),

        max("price").alias(
            "maximum_price"
        ),

        round(
            avg("horsepower"), 2
        ).alias(
            "average_horsepower"
        ),

        round(
            avg("city_mpg"), 2
        ).alias(
            "average_city_mpg"
        ),

        round(
            avg("highway_mpg"), 2
        ).alias(
            "average_highway_mpg"
        )

    )
    .withColumn(
        "gold_load_timestamp",
        current_timestamp()
    )
)

(
    df_gold_kpis.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        gold_kpis_table
    )
)

In [0]:
df_gold_by_make = (

    df_silver

    .groupBy(
        "manufacturer_id",
        "make"
    )

    .agg(

        count("*").alias(
            "vehicle_count"
        ),

        round(
            avg("price"), 2
        ).alias(
            "average_price"
        ),

        round(
            avg("horsepower"), 2
        ).alias(
            "average_horsepower"
        ),

        round(
            avg("city_mpg"), 2
        ).alias(
            "average_city_mpg"
        ),

        round(
            avg("highway_mpg"), 2
        ).alias(
            "average_highway_mpg"
        )

    )

    .withColumn(
        "gold_load_timestamp",
        current_timestamp()
    )

)

(
    df_gold_by_make.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        gold_by_make_table
    )
)

In [0]:
df_gold_by_fuel = (

    df_silver

    .groupBy(
        "fuel_type"
    )

    .agg(

        count("*").alias(
            "vehicle_count"
        ),

        round(
            avg("price"), 2
        ).alias(
            "average_price"
        ),

        round(
            avg("horsepower"), 2
        ).alias(
            "average_horsepower"
        ),

        round(
            avg("city_mpg"), 2
        ).alias(
            "average_city_mpg"
        ),

        round(
            avg("highway_mpg"), 2
        ).alias(
            "average_highway_mpg"
        )

    )

    .withColumn(
        "gold_load_timestamp",
        current_timestamp()
    )

)

(
    df_gold_by_fuel.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        gold_by_fuel_table
    )
)

In [0]:
df_gold_top_expensive = (

    df_silver

    .select(
        "automobile_id",
        "make",
        "fuel_type",
        "body_style",
        "horsepower",
        "city_mpg",
        "highway_mpg",
        "price"
    )

    .where(
        "price IS NOT NULL"
    )

    .orderBy(
        desc("price")
    )

    .limit(20)

    .withColumn(
        "gold_load_timestamp",
        current_timestamp()
    )

)

(
    df_gold_top_expensive.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        gold_top_expensive_table
    )
)

In [0]:
print(
    spark.table(
        gold_kpis_table
    ).count()
)

print(
    spark.table(
        gold_by_make_table
    ).count()
)

print(
    spark.table(
        gold_by_fuel_table
    ).count()
)

print(
    spark.table(
        gold_top_expensive_table
    ).count()
)

1
22
2
20
